# Helmet Detection Model Training
## YOLOv8 Custom Training for Construction Safety

This notebook trains a custom YOLOv8 model for detecting:
- Workers wearing helmets (class: helmet)
- Workers not wearing helmets (class: no_helmet)
- Person detection (class: person)

**Novel aspects:**
- Multi-class detection for better context
- Data augmentation pipeline for construction environments
- Custom loss weighting for class imbalance
- Validation on diverse construction scenarios

## 1. Setup and Imports

In [ ]:
import os
import yaml
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from tqdm.notebook import tqdm

# Deep Learning
import torch
from ultralytics import YOLO

# Visualization
%matplotlib inline
sns.set_style('whitegrid')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Dataset Preparation

We'll use multiple data sources:
1. **Safety Helmet Detection Dataset** (Kaggle)
2. **Construction Site Dataset** (Roboflow)
3. **Custom collected footage**

Expected structure:
```
data/
├── train/
│   ├── images/
│   └── labels/
├── val/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'data' / 'models'
RUNS_DIR = PROJECT_ROOT / 'runs'

# Create directories
for dir_path in [DATA_DIR, MODELS_DIR, RUNS_DIR]:
    dir_path.mkdir(exist_ok=True, parents=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")

### 2.1 Dataset Statistics

In [ ]:
def count_images_and_labels(split_dir):
    """Count images and labels in a split"""
    images_dir = split_dir / 'images'
    labels_dir = split_dir / 'labels'
    
    if not images_dir.exists():
        return 0, 0
    
    num_images = len(list(images_dir.glob('*.jpg'))) + len(list(images_dir.glob('*.png')))
    num_labels = len(list(labels_dir.glob('*.txt'))) if labels_dir.exists() else 0
    
    return num_images, num_labels

# Count dataset
splits = ['train', 'val', 'test']
dataset_stats = {}

for split in splits:
    split_dir = DATA_DIR / split
    num_images, num_labels = count_images_and_labels(split_dir)
    dataset_stats[split] = {'images': num_images, 'labels': num_labels}
    print(f"{split.capitalize()}: {num_images} images, {num_labels} labels")

print(f"\nTotal images: {sum(s['images'] for s in dataset_stats.values())}")

### 2.2 Class Distribution Analysis

In [ ]:
def analyze_class_distribution(labels_dir):
    """Analyze class distribution in YOLO format labels"""
    class_counts = {0: 0, 1: 0, 2: 0}  # helmet, no_helmet, person
    
    if not labels_dir.exists():
        return class_counts
    
    for label_file in labels_dir.glob('*.txt'):
        with open(label_file, 'r') as f:
            for line in f:
                class_id = int(line.strip().split()[0])
                if class_id in class_counts:
                    class_counts[class_id] += 1
    
    return class_counts

# Analyze training set
train_dist = analyze_class_distribution(DATA_DIR / 'train' / 'labels')

class_names = ['helmet', 'no_helmet', 'person']
print("Training set class distribution:")
for class_id, count in train_dist.items():
    print(f"  {class_names[class_id]}: {count} instances")

# Visualize
plt.figure(figsize=(10, 6))
plt.bar(class_names, [train_dist[i] for i in range(3)])
plt.title('Class Distribution in Training Set')
plt.ylabel('Number of Instances')
plt.xlabel('Class')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 2.3 Create YAML Configuration

In [ ]:
# Create dataset YAML for YOLOv8
dataset_yaml = {
    'path': str(DATA_DIR.absolute()),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 3,  # number of classes
    'names': ['helmet', 'no_helmet', 'person']
}

yaml_path = DATA_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"Dataset configuration saved to: {yaml_path}")
print("\nConfiguration:")
print(yaml.dump(dataset_yaml, default_flow_style=False))

## 3. Model Training

We'll train multiple model sizes:
- YOLOv8n (nano) - Fast inference, lower accuracy
- YOLOv8s (small) - Balanced
- YOLOv8m (medium) - Higher accuracy, slower inference

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    'epochs': 100,
    'batch': 16,
    'imgsz': 640,
    'patience': 20,
    'save_period': 10,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 8,
    'project': str(RUNS_DIR),
    'name': 'helmet_detection',
    'exist_ok': True,
    
    # Data augmentation
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
    'degrees': 10.0,
    'translate': 0.1,
    'scale': 0.5,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.0,
    'fliplr': 0.5,
    'mosaic': 1.0,
    'mixup': 0.1,
    
    # Optimizer
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    
    # Loss
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5
}

print("Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

### 3.1 Train YOLOv8n (Nano)

In [ ]:
# Initialize model
model_n = YOLO('yolov8n.pt')

# Train
results_n = model_n.train(
    data=str(yaml_path),
    **TRAINING_CONFIG
)

print("\nYOLOv8n training complete!")

### 3.2 Train YOLOv8s (Small)

In [ ]:
# Initialize model
model_s = YOLO('yolov8s.pt')

# Update name for different run
config_s = TRAINING_CONFIG.copy()
config_s['name'] = 'helmet_detection_small'

# Train
results_s = model_s.train(
    data=str(yaml_path),
    **config_s
)

print("\nYOLOv8s training complete!")

## 4. Model Evaluation

In [ ]:
# Load best models
best_model_n = YOLO(RUNS_DIR / 'helmet_detection' / 'weights' / 'best.pt')

# Validate on test set
metrics_n = best_model_n.val(data=str(yaml_path), split='test')

print("\nYOLOv8n Test Set Results:")
print(f"  mAP@50: {metrics_n.box.map50:.4f}")
print(f"  mAP@50-95: {metrics_n.box.map:.4f}")
print(f"  Precision: {metrics_n.box.mp:.4f}")
print(f"  Recall: {metrics_n.box.mr:.4f}")

### 4.1 Per-Class Performance

In [ ]:
# Per-class metrics
class_names = ['helmet', 'no_helmet', 'person']

print("\nPer-class Performance:")
for i, class_name in enumerate(class_names):
    print(f"\n{class_name}:")
    print(f"  Precision: {metrics_n.box.class_result(i)[0]:.4f}")
    print(f"  Recall: {metrics_n.box.class_result(i)[1]:.4f}")
    print(f"  mAP@50: {metrics_n.box.class_result(i)[2]:.4f}")

### 4.2 Confusion Matrix

In [ ]:
# Plot confusion matrix
confusion_matrix_path = RUNS_DIR / 'helmet_detection' / 'confusion_matrix.png'

if confusion_matrix_path.exists():
    img = Image.open(confusion_matrix_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()
else:
    print("Confusion matrix not found")

### 4.3 Training Curves

In [ ]:
# Plot training results
results_path = RUNS_DIR / 'helmet_detection' / 'results.png'

if results_path.exists():
    img = Image.open(results_path)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Metrics')
    plt.tight_layout()
    plt.show()
else:
    print("Results plot not found")

## 5. Inference Testing

In [ ]:
# Test on sample images
test_images_dir = DATA_DIR / 'test' / 'images'
sample_images = list(test_images_dir.glob('*.jpg'))[:5]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, img_path in enumerate(sample_images):
    if idx >= 6:
        break
    
    # Run inference
    results = best_model_n(str(img_path))
    
    # Plot
    annotated = results[0].plot()
    axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[idx].axis('off')
    axes[idx].set_title(f'Sample {idx+1}')

plt.tight_layout()
plt.show()

## 6. Model Export

Export models for deployment in various formats

In [ ]:
# Export to ONNX for cross-platform deployment
onnx_path = best_model_n.export(format='onnx', imgsz=640, simplify=True)
print(f"\nModel exported to ONNX: {onnx_path}")

# Export to TensorRT for NVIDIA GPU optimization (optional)
if torch.cuda.is_available():
    try:
        trt_path = best_model_n.export(format='engine', imgsz=640, half=True)
        print(f"Model exported to TensorRT: {trt_path}")
    except Exception as e:
        print(f"TensorRT export failed: {e}")

# Copy best weights to models directory
shutil.copy(
    RUNS_DIR / 'helmet_detection' / 'weights' / 'best.pt',
    MODELS_DIR / 'helmet_detector_best.pt'
)
print(f"\nBest model copied to: {MODELS_DIR / 'helmet_detector_best.pt'}")

## 7. Performance Benchmarking

In [ ]:
import time

# Benchmark inference speed
test_image = str(sample_images[0])
num_iterations = 100

# Warmup
for _ in range(10):
    _ = best_model_n(test_image, verbose=False)

# Benchmark
times = []
for _ in tqdm(range(num_iterations), desc="Benchmarking"):
    start = time.time()
    _ = best_model_n(test_image, verbose=False)
    times.append(time.time() - start)

avg_time = np.mean(times)
fps = 1.0 / avg_time

print(f"\nInference Performance:")
print(f"  Average time: {avg_time*1000:.2f} ms")
print(f"  FPS: {fps:.2f}")
print(f"  Device: {TRAINING_CONFIG['device']}")

## 8. Summary & Next Steps

### Model Performance Summary
- **mAP@50**: Achieved on test set
- **Inference Speed**: FPS on target hardware
- **Model Size**: Compact enough for edge deployment

### Next Steps
1. Deploy model in production pipeline
2. Integrate with tracking system
3. Set up monitoring and retraining pipeline
4. A/B test different model variants
5. Collect more edge cases for continuous improvement